In [ ]:
import torch

import gpt2

print(torch.__version__)
print(gpt2.__file__)

## Linear layer: `y = Wx + b`

A single "layer" in a neural net takes an input vector `x`, multiplies it by a weight matrix `W`, and adds a bias vector `b`:

```
y = W @ x + b
```

We'll use:
- 3 input neurons → `x` has shape `(3,)`
- 2 output neurons → `y` has shape `(2,)`

For that to work, `W` must have shape `(2, 3)` — **(out_features, in_features)**. Matrix-vector multiply `(2,3) @ (3,) -> (2,)`. Each output neuron is a weighted sum of *all* input neurons, plus its own bias.

Let's build it with plain numpy first, one piece at a time.

In [1]:
import numpy as np

# 3 input neurons -> a single activation value per neuron
x = np.array([1.0, 2.0, 3.0])
print("x shape:", x.shape)
print("x:", x)

x shape: (3,)
x: [1. 2. 3.]


In [7]:
# Weight matrix: one row per output neuron, one column per input neuron
# row 0 = weights feeding output neuron 0, row 1 = weights feeding output neuron 1
W = np.array([
    [0.1, 0.2, 0.3],   # weights for output neuron 0
    [0.4, 0.5, 0.6],   # weights for output neuron 1
])
print("W shape:", W.shape)  # (2, 3) = (out_features, in_features)

b = np.array([0.01, 0.02])
print("b shape:", b.shape)  # (2,) one bias per output neuron

W shape: (2, 3)
b shape: (2,)


In [9]:
a1 = W @ x
print("a1 shape:", a1.shape)  # (2,) one activation per output neuron
print("a1:", a1)
a2 = a1 + b
print("a2 shape:", a2.shape)  # (2,) one activation per output neuron
print("a2:", a2)

a1 shape: (2,)
a1: [1.4 3.2]
a2 shape: (2,)
a2: [1.41 3.22]


In [10]:
# Manual version: for every output neuron, take the dot product of its
# weight row with x, then add its bias.
y_manual = np.zeros(2)
for out_idx in range(W.shape[0]):        # 2 output neurons
    total = 0.0
    for in_idx in range(W.shape[1]):     # 3 input neurons
        total += W[out_idx, in_idx] * x[in_idx]
    y_manual[out_idx] = total + b[out_idx]

print("y (manual loops):", y_manual)

y (manual loops): [1.41 3.22]


In [11]:
# Same thing, the real way: matrix @ vector + vector
y = W @ x + b
print("y (W @ x + b):", y)

assert np.allclose(y, y_manual)
print("matches manual computation ✓")

y (W @ x + b): [1.41 3.22]
matches manual computation ✓


## Batching: many inputs at once

In practice you never push one vector through a layer — you push a **batch** of `N` samples together, stacked into a matrix `X` of shape `(N, in_features)`.

Convention: each **row** of `X` is one sample. So for `N=4` samples of our 3-input data, `X` has shape `(4, 3)`.

The layer becomes:

```
Y = X @ W.T + b
```

Shapes: `(N, 3) @ (3, 2) -> (N, 2)`, then `b` (shape `(2,)`) broadcasts and gets added to every row. Note the `W.T` — we transpose `W` from `(out, in)` to `(in, out)` so the matmul lines up.

In [12]:
# 2 samples, each with 3 input features (one row per sample)
X = np.array([
    [1.0, 2.0, 3.0],   # sample 0 -- same as our earlier x
    [0.0, 0.0, 1.0],   # sample 1 -- a different, unrelated input
])
print("X shape:", X.shape)  # (2, 3) = (N, in_features)

Y = X @ W.T + b
print("Y shape:", Y.shape)  # (2, 2) = (N, out_features)
print(Y)

# sanity check: row 0 of X is exactly our earlier x, so row 0 of Y should match y
print("row 0 matches single-vector result:", np.allclose(Y[0], y))

X shape: (2, 3)
Y shape: (2, 2)
[[1.41 3.22]
 [0.31 0.62]]
row 0 matches single-vector result: True


## Adding a hidden layer

Now: 3 input neurons → **4 hidden neurons** → 2 output neurons.

Each layer is still just `y = Wx + b`, but now there are two of them chained together, each with its own weight matrix and bias:

- Layer 1 (input → hidden): `W1` shape `(4, 3)`, `b1` shape `(4,)`. Takes `x` (3,) → produces `h` (4,).
- Layer 2 (hidden → output): `W2` shape `(2, 4)`, `b2` shape `(2,)`. Takes `h` (4,) → produces `y` (2,).

Notice: `W2`'s number of *columns* (4) must match `W1`'s number of *rows* (4) — the hidden layer's output size. That's the chain: `3 → 4 → 2`.

One catch: if you just did `y = W2 @ (W1 @ x + b1) + b2` with nothing in between, that whole thing algebraically collapses into a *single* equivalent linear layer (a matrix product of matrices is still just a matrix). Stacking layers only gains you anything if you put a **nonlinear activation function** between them — we'll use `ReLU(z) = max(0, z)`, applied elementwise to the hidden layer's output.

In [13]:
# Layer 1: input (3) -> hidden (4)
W1 = np.array([
    [0.1, 0.2, 0.3],
    [0.4, 0.5, 0.6],
    [0.7, 0.8, 0.9],
    [1.0, 1.1, 1.2],
])
b1 = np.array([0.01, 0.02, 0.03, 0.04])
print("W1 shape:", W1.shape)  # (4, 3) = (hidden_features, in_features)
print("b1 shape:", b1.shape)  # (4,)

h_pre = W1 @ x + b1
print("h_pre (before activation):", h_pre, "shape:", h_pre.shape)

h = np.maximum(0, h_pre)   # ReLU, applied elementwise
print("h (after ReLU):", h)

W1 shape: (4, 3)
b1 shape: (4,)
h_pre (before activation): [1.41 3.22 5.03 6.84] shape: (4,)
h (after ReLU): [1.41 3.22 5.03 6.84]


In [14]:
# Layer 2: hidden (4) -> output (2)
W2 = np.array([
    [0.1, 0.2, 0.3, 0.4],
    [0.5, 0.6, 0.7, 0.8],
])
b2 = np.array([0.01, 0.02])
print("W2 shape:", W2.shape)  # (2, 4) = (out_features, hidden_features)
print("b2 shape:", b2.shape)  # (2,)

y_final = W2 @ h + b2
print("y_final:", y_final, "shape:", y_final.shape)  # (2,) -- back to our 2 output neurons

W2 shape: (2, 4)
b2 shape: (2,)
y_final: [ 5.04 11.65] shape: (2,)
